# DQN pour la décision de changement de voie (SUMO + Stable-Baselines3)

**Avant de lancer :** mettre les 4 fichiers du dossier `data/` (`obstacles.net.xml`, `obstacles.rou.xml`,
`obstacles.sumocfg`, `obstacles.view.xml`) dans `MyDrive/data` sur Google Drive.
Un runtime **CPU** suffit (petit réseau, le goulot d'étranglement est SUMO).
Exécuter les cellules dans l'ordre (*Exécution > Tout exécuter*).

Corrections par rapport à l'ancienne version :
- les véhicules rouges sont **immobiles** (comme demandé dans l'énoncé) → l'agent rencontre enfin des obstacles ;
- la récompense ne récompense plus les actions « changer de voie » inutiles ou impossibles
  (l'ancien agent gagnait +5 en demandant une voie inexistante) ; chaque changement de voie coûte -1,
  donc l'agent ne change de voie que pour éviter un obstacle ;
- sortir de la route = **collision avec le bord** (pénalité + fin d'épisode) ; arriver au bout de la route = **succès** (bonus) ;
- observation **normalisée** dans [0, 1] ;
- l'évaluation compare le DQN et le modèle SUMO par défaut **dans les mêmes conditions**.

In [ ]:
# CELLULE 1 - Installation (SUMO via pip : pas besoin d'apt-get)
!pip install -q eclipse-sumo traci sumolib "stable-baselines3>=2.0" gymnasium tensorboard

In [ ]:
# CELLULE 2 - Chemins + Google Drive
import os
try:                      # SUMO installé par pip (eclipse-sumo)
    import sumo
    os.environ["SUMO_HOME"] = sumo.SUMO_HOME
except ImportError:       # sinon : !apt-get install -y sumo sumo-tools
    os.environ.setdefault("SUMO_HOME", "/usr/share/sumo")

from google.colab import drive
drive.mount("/content/drive")

DATA_DIR = "/content/drive/MyDrive/data"            # fichiers SUMO
OUT_DIR = "/content/drive/MyDrive/ProjetRLOC_DQN"   # modèle, logs, figures, CSV (sauvegardés sur Drive)

for f in ["obstacles.net.xml", "obstacles.rou.xml", "obstacles.sumocfg"]:
    assert os.path.exists(os.path.join(DATA_DIR, f)), f"Fichier manquant : {DATA_DIR}/{f}"
for d in ["models", "runs", "plots", "results"]:
    os.makedirs(os.path.join(OUT_DIR, d), exist_ok=True)
print("SUMO_HOME =", os.environ["SUMO_HOME"])
print("Données OK, sorties dans", OUT_DIR)

In [ ]:
# CELLULE 3 - Environnement SUMO (Gymnasium)
"""
Environnement SUMO commun (Gymnasium) pour la décision de changement de voie.

Scénario (cf. énoncé) : autoroute à 2 voies, véhicules rouges IMMOBILES servant
d'obstacles, véhicule autonome jaune (AV) dont la vitesse est gérée par SUMO.
L'agent ne décide que du changement de voie :

    0 = garder la voie, 1 = voie de gauche, 2 = voie de droite

Convention SUMO : l'index de voie 0 est la voie la plus à DROITE,
donc "gauche" = index + 1 et "droite" = index - 1.

Observation (continue, dans [0, 1]) :
    [vitesse / v_max, gap_courant / R, gap_gauche / R, gap_droite / R]
    gap = distance libre devant l'AV dans la voie considérée (plafonnée à R).
    Une voie inexistante (bord de route) ou occupée à côté de l'AV vaut 0 :
    c'est un "mur" contre lequel un changement de voie provoquerait une collision.

Récompense par pas (1 pas = 1 s) :
    + v / v_max - 0.2           progression moins un coût de temps
                                (maximiser la vitesse = minimiser le temps de parcours)
    - 1.0                       si l'AV a effectivement changé de voie (plus qu'un pas de progression :
                                un changement n'est rentable que s'il évite un obstacle)
    - 50  + fin d'épisode       collision avec le bord de la route ou un obstacle
                                (toujours pire que rester bloqué jusqu'à la fin : 200 x 0.2 = 40)
    + 10  + fin d'épisode       arrivée au bout de la route
L'épisode est tronqué après MAX_STEPS pas (AV bloqué derrière un obstacle).
"""

import os
import itertools

import numpy as np
import gymnasium as gym
from gymnasium import spaces

if "SUMO_HOME" in os.environ:
    import sys
    sys.path.append(os.path.join(os.environ["SUMO_HOME"], "tools"))
import traci  # noqa: E402
import sumolib  # noqa: E402

SUMO_CFG = os.path.join(DATA_DIR, "obstacles.sumocfg")

EGO_ID = "vehAgent"
ROUTE_ID = "r_0"
EGO_TYPE = "obstacle"          # même vType que dans run.py fourni
STEP_LENGTH = 1.0              # s
MAX_STEPS = 200                # parcours libre ~ 80 pas
GAP_RANGE = 100.0              # m, portée de perception
TIME_PENALTY = 0.2
LANE_CHANGE_COST = 1.0
USELESS_GAP = GAP_RANGE        # changement de voie sans aucun obstacle visible devant (100 m libres) = inutile
COLLISION_PENALTY = 50.0
ARRIVAL_BONUS = 10.0

KEEP, LEFT, RIGHT = 0, 1, 2
ACTION_NAMES = {KEEP: "keep", LEFT: "left", RIGHT: "right"}


_conn_counter = itertools.count()


def sumo_binary(gui=False):
    name = "sumo-gui" if gui else "sumo"
    try:
        return sumolib.checkBinary(name)
    except Exception:
        return name



def greedy_rollouts(env, policy, n_episodes, seed0):
    """Joue n épisodes avec une politique déterministe ; renvoie (récompense totale moyenne, taux d'arrivée)."""
    rewards, arrived = [], []
    for k in range(n_episodes):
        obs, _ = env.reset(seed=seed0 + k)
        done = False
        while not done:
            obs, _, terminated, truncated, info = env.step(policy(obs))
            done = terminated or truncated
        rewards.append(info["total_reward"])
        arrived.append(info["arrived"])
    return float(np.mean(rewards)), float(np.mean(arrived))


class LaneChangeEnv(gym.Env):
    metadata = {"render_modes": ["human"]}

    def __init__(self, gui=False, sumo_default=False, randomize_start=True,
                 max_steps=MAX_STEPS, gui_delay=100):
        """
        gui            : lance sumo-gui au lieu de sumo
        sumo_default   : l'AV utilise le modèle de changement de voie par défaut de SUMO
                         (les actions de l'agent sont ignorées) -> baseline
        randomize_start: position / voie / vitesse de départ aléatoires (sinon départ fixe)
        """
        super().__init__()
        self.gui = gui
        self.sumo_default = sumo_default
        self.randomize_start = randomize_start
        self.max_steps = max_steps
        self.gui_delay = gui_delay

        self.action_space = spaces.Discrete(3)
        self.observation_space = spaces.Box(0.0, 1.0, shape=(4,), dtype=np.float32)

        self.label = f"lcenv_{os.getpid()}_{next(_conn_counter)}"
        self.conn = None
        self.obstacles = {}  # lane index -> liste triée de (back, front)
        self.v_max = 13.89
        self.ego_len = 5.0
        self.n_lanes = 2

    # ------------------------------------------------------------------ SUMO
    def _start_sumo(self):
        """Lance SUMO au premier reset, puis recharge simplement la simulation
        (traci.load) : beaucoup plus rapide que de relancer un processus à chaque épisode."""
        args = [
            "-c", SUMO_CFG,
            "--step-length", str(STEP_LENGTH),
            "--collision.action", "warn",
            "--collision.check-junctions", "true",
            "--time-to-teleport", "-1",
            "--xml-validation", "never",
            "--no-step-log", "true",
            "--no-warnings", "true",
            "--seed", str(int(self.np_random.integers(0, 2**31 - 1))),
        ]
        if self.gui:
            args += ["--start", "--quit-on-end", "--delay", str(self.gui_delay)]
        if self.conn is None:
            traci.start([sumo_binary(self.gui)] + args, label=self.label)
            self.conn = traci.getConnection(self.label)
        else:
            self.conn.load(args)

    def close(self):
        if self.conn is not None:
            try:
                self.conn.close()
            except Exception:
                pass
            self.conn = None

    # ----------------------------------------------------------------- reset
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self._start_sumo()
        c = self.conn

        # 1) insérer les obstacles puis les immobiliser (comme dans run.py)
        c.simulationStep()
        for v in c.vehicle.getIDList():
            c.vehicle.setLaneChangeMode(v, 0)
            c.vehicle.setSpeed(v, 0)

        # 2) insérer l'AV
        if self.randomize_start:
            depart_pos = float(self.np_random.uniform(0.0, 250.0))
            depart_lane = int(self.np_random.integers(0, 2))
            depart_speed = float(self.np_random.uniform(0.0, 13.0))
        else:
            depart_pos, depart_lane, depart_speed = 0.0, 0, 13.0
        c.vehicle.add(EGO_ID, ROUTE_ID, typeID=EGO_TYPE, depart="now",
                      departPos=f"{depart_pos:.2f}", departLane=str(depart_lane),
                      departSpeed=f"{depart_speed:.2f}")
        c.vehicle.setColor(EGO_ID, (255, 255, 0, 255))
        if not self.sumo_default:
            c.vehicle.setLaneChangeMode(EGO_ID, 0)  # désactive le modèle LC de SUMO
        for _ in range(20):
            c.simulationStep()
            if EGO_ID in c.vehicle.getIDList():
                break
        else:
            raise RuntimeError("L'AV n'a pas pu être inséré dans SUMO")

        # 3) les obstacles sont immobiles : on mémorise leurs positions une fois pour toutes
        self.obstacles = {}
        for v in c.vehicle.getIDList():
            if v == EGO_ID:
                continue
            front = c.vehicle.getLanePosition(v)
            length = c.vehicle.getLength(v)
            self.obstacles.setdefault(c.vehicle.getLaneIndex(v), []).append((front - length, front))
        for lst in self.obstacles.values():
            lst.sort()

        edge = c.vehicle.getRoadID(EGO_ID)
        self.n_lanes = c.edge.getLaneNumber(edge)
        self.v_max = c.lane.getMaxSpeed(c.vehicle.getLaneID(EGO_ID))
        self.ego_len = c.vehicle.getLength(EGO_ID)
        self.edge_len = c.lane.getLength(c.vehicle.getLaneID(EGO_ID))

        self.steps = 0
        self.total_reward = 0.0
        self.lane_changes = 0
        self.useless_lane_changes = 0
        self.boundary_collisions = 0
        self.vehicle_collisions = 0
        self.speed_sum = 0.0
        self.start_pos = c.vehicle.getLanePosition(EGO_ID)
        self.lane = c.vehicle.getLaneIndex(EGO_ID)
        self.pos = self.start_pos
        self.speed = c.vehicle.getSpeed(EGO_ID)
        return self._obs(), self._info(arrived=False)

    # ----------------------------------------------------------- observation
    def _gap(self, lane):
        """Distance libre devant l'AV dans `lane` (0 si voie inexistante ou occupée à côté)."""
        if lane < 0 or lane >= self.n_lanes:
            return 0.0
        ego_front, ego_back = self.pos, self.pos - self.ego_len
        gap = GAP_RANGE
        for back, front in self.obstacles.get(lane, []):
            if front <= ego_back:
                continue                     # obstacle derrière l'AV
            gap = min(gap, max(0.0, back - ego_front))  # 0 si chevauchement latéral
            break                            # liste triée : premier obstacle devant
        return gap

    def _obs(self):
        return np.array([
            min(self.speed / self.v_max, 1.0),
            self._gap(self.lane) / GAP_RANGE,
            self._gap(self.lane + 1) / GAP_RANGE,   # gauche
            self._gap(self.lane - 1) / GAP_RANGE,   # droite
        ], dtype=np.float32)

    def _info(self, arrived):
        return {
            "steps": self.steps,
            "travel_time": self.steps * STEP_LENGTH,
            "arrived": arrived,
            "lane_changes": self.lane_changes,
            "useless_lane_changes": self.useless_lane_changes,
            "boundary_collisions": self.boundary_collisions,
            "vehicle_collisions": self.vehicle_collisions,
            "distance": self.pos - self.start_pos,
            "mean_speed": self.speed_sum / max(self.steps, 1),
            "total_reward": self.total_reward,
        }

    # ------------------------------------------------------------------ step
    def step(self, action):
        c = self.conn
        action = int(action)
        reward = 0.0
        terminated = False
        arrived = False
        prev_lane = self.lane
        gap_before = self._gap(self.lane)

        # --- action de l'agent (ignorée si le modèle SUMO par défaut conduit) ---
        if not self.sumo_default and action != KEEP:
            target = self.lane + 1 if action == LEFT else self.lane - 1
            if 0 <= target < self.n_lanes:
                c.vehicle.changeLane(EGO_ID, target, STEP_LENGTH)
            else:
                # collision avec le bord de la route
                self.boundary_collisions += 1
                self.steps += 1
                reward = -COLLISION_PENALTY
                self.total_reward += reward
                return self._obs(), reward, True, False, self._info(arrived=False)

        c.simulationStep()
        self.steps += 1

        if EGO_ID not in c.vehicle.getIDList():
            # l'AV a quitté le réseau : fin de route atteinte
            arrived = True
            terminated = True
            last_step = max(self.edge_len - self.pos, 0.0)
            self.speed_sum += min(last_step / STEP_LENGTH, self.v_max)
            self.pos = self.edge_len
            reward = min(last_step / (self.v_max * STEP_LENGTH), 1.0) - TIME_PENALTY + ARRIVAL_BONUS
        else:
            self.lane = c.vehicle.getLaneIndex(EGO_ID)
            self.pos = c.vehicle.getLanePosition(EGO_ID)
            self.speed = c.vehicle.getSpeed(EGO_ID)
            self.speed_sum += self.speed
            reward = min(self.speed / self.v_max, 1.0) - TIME_PENALTY
            if self.lane != prev_lane:
                self.lane_changes += 1
                if gap_before >= USELESS_GAP:
                    self.useless_lane_changes += 1
                reward -= LANE_CHANGE_COST
            if EGO_ID in c.simulation.getCollidingVehiclesIDList():
                self.vehicle_collisions += 1
                reward -= COLLISION_PENALTY
                terminated = True

        truncated = (not terminated) and self.steps >= self.max_steps
        self.total_reward += reward
        return self._obs(), reward, terminated, truncated, self._info(arrived)

In [ ]:
# CELLULE 4 - Test rapide de l'environnement
# "toujours garder la voie" doit rester bloqué derrière un obstacle,
# le modèle SUMO par défaut doit arriver au bout de la route.
for name, sumo_default in [("garder la voie", False), ("SUMO par défaut", True)]:
    env = LaneChangeEnv(sumo_default=sumo_default)
    obs, _ = env.reset(seed=0)
    done = False
    while not done:
        obs, r, terminated, truncated, info = env.step(KEEP)
        done = terminated or truncated
    env.close()
    print(f"{name:16s} | arrivé={info['arrived']} | temps={info['travel_time']:.0f} s | "
          f"distance={info['distance']:.0f} m | récompense totale={info['total_reward']:.1f}")

In [ ]:
# CELLULE 5 - Entraînement DQN + logs TensorBoard
import pandas as pd
import matplotlib.pyplot as plt
from torch.utils.tensorboard import SummaryWriter
from stable_baselines3 import DQN
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.monitor import Monitor

TOTAL_TIMESTEPS = 150_000
EVAL_EVERY_STEPS, EVAL_EPISODES, EVAL_SEED = 2_500, 5, 500_000


class EpisodeLogger(BaseCallback):
    """Écrit la récompense totale (non actualisée) de chaque épisode dans TensorBoard
    et évalue régulièrement la politique greedy (sans exploration)."""

    def __init__(self, log_dir, eval_env):
        super().__init__()
        self.writer = SummaryWriter(log_dir)
        self.eval_env = eval_env
        self.history, self.eval_history = [], []
        self.last_eval = 0

    def _on_step(self):
        for info, done in zip(self.locals["infos"], self.locals["dones"]):
            if not done:
                continue
            ep = len(self.history)
            row = {"episode": ep, "timestep": self.num_timesteps,
                   **{k: info[k] for k in ("total_reward", "travel_time", "arrived", "lane_changes", "useless_lane_changes",
                                           "boundary_collisions", "vehicle_collisions")}}
            self.history.append(row)
            self.writer.add_scalar("episode/total_reward", row["total_reward"], ep)
            self.writer.add_scalar("episode/travel_time", row["travel_time"], ep)
            self.writer.add_scalar("episode/lane_changes", row["lane_changes"], ep)
            self.writer.add_scalar("episode/useless_lane_changes", row["useless_lane_changes"], ep)
            self.writer.add_scalar("episode/arrived", float(row["arrived"]), ep)
            if self.num_timesteps - self.last_eval >= EVAL_EVERY_STEPS:
                self.last_eval = self.num_timesteps
                policy = lambda o: int(self.model.predict(o, deterministic=True)[0])
                mean_r, arr = greedy_rollouts(self.eval_env, policy, EVAL_EPISODES, EVAL_SEED)
                self.writer.add_scalar("eval/total_reward", mean_r, ep)
                self.writer.add_scalar("eval/arrival_rate", arr, ep)
                self.eval_history.append({"episode": ep, "timestep": self.num_timesteps,
                                          "eval_total_reward": mean_r, "eval_arrival_rate": arr})
                print(f"pas {self.num_timesteps:6d} | épisode {ep:5d} | "
                      f"politique greedy : récompense={mean_r:6.1f}, arrivées={arr:.0%}")
        return True

    def _on_training_end(self):
        self.writer.close()


env = Monitor(LaneChangeEnv())
env.reset(seed=0)
eval_env = LaneChangeEnv()

model = DQN(
    "MlpPolicy", env,
    learning_rate=5e-4,
    buffer_size=50_000,
    learning_starts=2_000,
    batch_size=64,
    gamma=0.95,
    train_freq=4,
    gradient_steps=1,
    target_update_interval=1_000,
    exploration_fraction=0.3,
    exploration_initial_eps=1.0,
    exploration_final_eps=0.005,
    policy_kwargs=dict(net_arch=[64, 64]),   # MLP 4 -> 64 -> 64 -> 3
    tensorboard_log=os.path.join(OUT_DIR, "runs"),
    seed=0,
    verbose=0,
)

callback = EpisodeLogger(os.path.join(OUT_DIR, "runs", "dqn_episodes"), eval_env)
model.learn(total_timesteps=TOTAL_TIMESTEPS, callback=callback, tb_log_name="dqn")
model.save(os.path.join(OUT_DIR, "models", "dqn_lanechange"))
env.close()
eval_env.close()

train_df = pd.DataFrame(callback.history)
eval_df = pd.DataFrame(callback.eval_history)
train_df.to_csv(os.path.join(OUT_DIR, "results", "dqn_training.csv"), index=False)
eval_df.to_csv(os.path.join(OUT_DIR, "results", "dqn_eval_during_training.csv"), index=False)
print("✅ Modèle sauvegardé :", os.path.join(OUT_DIR, "models", "dqn_lanechange.zip"))

In [ ]:
# CELLULE 6 - TensorBoard (onglet SCALARS : episode/total_reward, eval/total_reward, train/loss...)
%load_ext tensorboard
%tensorboard --logdir "$OUT_DIR/runs"

In [ ]:
# CELLULE 7 - Courbe de convergence
plt.figure(figsize=(9, 5))
plt.plot(train_df.episode, train_df.total_reward, alpha=0.3, label="récompense totale / épisode")
plt.plot(train_df.episode, train_df.total_reward.rolling(50, min_periods=1).mean(),
         label="moyenne glissante (50)")
plt.plot(eval_df.episode, eval_df.eval_total_reward, "k.-", label="politique greedy (moy. 5 épisodes)")
plt.xlabel("Épisode")
plt.ylabel("Récompense totale (non actualisée)")
plt.title("DQN - convergence")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "plots", "dqn_training_reward.png"), dpi=120)
plt.show()

In [ ]:
# CELLULE 8 - Évaluation : DQN vs modèle de changement de voie par défaut de SUMO
# Mêmes 50 conditions de départ (seeds jamais vues à l'entraînement) pour les deux politiques.
# Métriques : temps de parcours (objectif du projet ; échec = MAX_STEPS s) et nombre de changements de voie
# (dont « inutiles » = changement alors qu'aucun obstacle n'est visible devant, 100 m libres).
import numpy as np

model = DQN.load(os.path.join(OUT_DIR, "models", "dqn_lanechange.zip"), device="cpu")
POLICIES = {
    "DQN": (lambda o: int(model.predict(o, deterministic=True)[0]), False),
    "SUMO_default": (lambda o: KEEP, True),   # actions ignorées : SUMO décide
}
N_EVAL = 50
rows = []
for name, (policy, sumo_default) in POLICIES.items():
    env = LaneChangeEnv(sumo_default=sumo_default)
    for ep in range(N_EVAL):
        obs, _ = env.reset(seed=1_000_000 + ep)
        done = False
        while not done:
            obs, r, terminated, truncated, info = env.step(policy(obs))
            done = terminated or truncated
        rows.append({"policy": name, "episode": ep + 1,
                     "travel_time": info["travel_time"] if info["arrived"] else float(MAX_STEPS),
                     "lane_changes": info["lane_changes"],
                     "useless_lane_changes": info["useless_lane_changes"],
                     "arrived": int(info["arrived"]),
                     "collisions": info["boundary_collisions"] + info["vehicle_collisions"],
                     "mean_speed": round(info["mean_speed"], 3),
                     "total_reward": round(info["total_reward"], 3)})
    env.close()

res = pd.DataFrame(rows)
res.to_csv(os.path.join(OUT_DIR, "results", "evaluation_dqn_vs_sumo.csv"), index=False)
summary = res.groupby("policy").agg(
    travel_time_mean=("travel_time", "mean"), travel_time_std=("travel_time", "std"),
    lane_changes_mean=("lane_changes", "mean"), lane_changes_std=("lane_changes", "std"),
    useless_lane_changes_mean=("useless_lane_changes", "mean"),
    arrival_rate=("arrived", "mean"), collision_rate=("collisions", "mean"),
    mean_speed=("mean_speed", "mean"), total_reward_mean=("total_reward", "mean")).round(2)
summary.to_csv(os.path.join(OUT_DIR, "results", "evaluation_summary.csv"))
summary

In [ ]:
# CELLULE 9 - Figures de comparaison
colors = {"DQN": "tab:blue", "SUMO_default": "tab:green"}
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, (metric, label) in zip(axes, [("travel_time", "Temps de parcours (s)"),
                                      ("lane_changes", "Nombre de changements de voie")]):
    names = list(POLICIES)
    data = [res[res.policy == p][metric] for p in names]
    ax.bar(names, [d.mean() for d in data], yerr=[d.std() for d in data],
           color=[colors[p] for p in names], capsize=6)
    ax.set_title(f"{label} (moyenne ± écart-type, {N_EVAL} épisodes)")
    ax.set_ylabel(label)
    ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "plots", "evaluation_dqn_vs_sumo.png"), dpi=120)
plt.show()

In [ ]:
# CELLULE 10 - Télécharger le modèle et les résultats sur ton PC
# (ils sont aussi déjà sauvegardés sur Google Drive dans MyDrive/ProjetRLOC_DQN)
import shutil
from google.colab import files

print("Fichiers produits :")
for root, _, names in os.walk(OUT_DIR):
    for n in names:
        if not n.startswith("events.out"):
            print("  ", os.path.relpath(os.path.join(root, n), OUT_DIR))

files.download(os.path.join(OUT_DIR, "models", "dqn_lanechange.zip"))   # le modèle seul
shutil.make_archive("/content/ProjetRLOC_DQN_resultats", "zip", OUT_DIR)
files.download("/content/ProjetRLOC_DQN_resultats.zip")                  # tout : modèle, logs, figures, CSV